# Predição de Bitcoin com Random Forest

Escolha da estudante: **Random Forest**. Os últimos **sete fechamentos diários consecutivos**, do mais antigo ao mais recente, servem de entrada para estimar o fechamento do dia seguinte em USD.

Fonte: Bitstamp via CryptoDataDownload; período de 01/01/2023 a 04/10/2026. Excluímos o dia corrente e sequências que atravessam a lacuna em 22/05/2026. O notebook e o container reutilizam `src/train.py`.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import joblib

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.train import load_data, make_samples, train
DATA = ROOT / 'data/processed/btcusd_daily.csv'
OUTPUT = ROOT / 'models'
data = load_data(DATA)
print(f'{len(data)} registros: {data.date.min().date()} a {data.date.max().date()}')
data.tail(7)

1372 registros: 2023-01-01 a 2026-10-04


,date,close,volume_btc
1365,2026-09-28,83461.65,1812.672423
1366,2026-09-29,83629.41,1350.227326
1367,2026-09-30,83562.58,1819.360610
1368,2026-10-01,84852.65,1410.059300
1369,2026-10-02,84500.84,1967.878680
1370,2026-10-03,84746.87,454.309318
1371,2026-10-04,86510.16,748.653954


## Preparação das entradas
Cada amostra contém sete fechamentos passados e tem como alvo o fechamento do dia seguinte. Verificamos os oito dias para impedir que a lacuna seja tratada como um dia comum. Não usamos volume nesta versão.

In [2]:
X, y, target_dates, skipped = make_samples(data)
print(f'Amostras utilizáveis: {len(X)}; sequências descartadas por lacunas: {skipped}')
example = X.head(3).copy()
example['target_date'] = target_dates[:3]
example['target_close'] = y[:3]
example

Amostras utilizáveis: 1358; sequências descartadas por lacunas: 7


,close_lag_7,close_lag_6,close_lag_5,close_lag_4,close_lag_3,close_lag_2,close_lag_1,target_date,target_close
0,16615.0,16673.0,16670.0,16849.0,16826.0,16950.0,16945.0,2023-01-08,17119.0
1,16673.0,16670.0,16849.0,16826.0,16950.0,16945.0,17119.0,2023-01-09,17179.0
2,16670.0,16849.0,16826.0,16950.0,16945.0,17119.0,17179.0,2023-01-10,17446.0


## Treinamento e avaliação cronológica
Os **80% mais antigos** ficam no treino e os **20% mais recentes** no teste, sem embaralhar. Avaliamos um dia à frente usando os sete fechamentos anteriores conhecidos; não é uma previsão recursiva de todo o teste.

Parâmetros fixos, sem ajuste no teste: 200 árvores, profundidade máxima 10, mínimo de 3 amostras por folha e semente 42. Comparamos MAE e RMSE com repetir o último fechamento.

Após a avaliação, ajustamos um **segundo modelo com todo o histórico** para exportar. As métricas abaixo pertencem exclusivamente ao modelo treinado nos 80% iniciais.

In [3]:
result = train(DATA, OUTPUT)
print(json.dumps({key: result['evaluation'][key] for key in ['train_samples', 'test_samples', 'train_target_last_date', 'test_target_first_date', 'test_target_last_date']}, indent=2))
pd.DataFrame({
    'Random Forest': result['evaluation']['random_forest'],
    'Repetir último fechamento': result['evaluation']['baseline_last_close'],
}).T

{
  "train_samples": 1086,
  "test_samples": 272,
  "train_target_last_date": "2025-12-28",
  "test_target_first_date": "2025-12-29",
  "test_target_last_date": "2026-10-04"
}


,mae_usd,rmse_usd
Random Forest,1678.020663,2223.823277
Repetir último fechamento,1183.965772,1742.506113


## Exportação e demonstração local
Geramos `models/model.joblib`, `models/metadata.json` e `models/test_predictions.csv`. Recarregamos o artefato para conferir a previsão sem treinar novamente. A demonstração por HTTP será feita na etapa do backend.

In [4]:
bundle = joblib.load(OUTPUT / 'model.joblib')
last_week = data.tail(7)
inputs = pd.DataFrame([last_week['close'].to_numpy()], columns=bundle['metadata']['feature_names'])
prediction = float(bundle['model'].predict(inputs)[0])
assert abs(prediction - result['example']['predicted_close_usd']) < 1e-6
print(f"Fechamento previsto para {result['example']['prediction_date']}: USD {prediction:,.2f}")
print('Artefato recarregado: previsão conferida.')

Fechamento previsto para 2026-10-05: USD 86,913.88
Artefato recarregado: previsão conferida.


## Limitações
Usamos apenas sete fechamentos, sem notícias ou outros fatores de mercado. Random Forest não extrapola bem para preços fora da faixa observada. Um resultado pior que o baseline deve ser documentado. A previsão é experimental para demonstrar treinamento, exportação e inferência.

Referência: [RandomForestRegressor — scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html).